In [1]:
import pandas as pd

path = "../data/raw/10.35097-1969/10.35097-1969/data/dataset/cell_eisv2/cell_eisv2_P001_1_S01_C10.csv"
df = pd.read_csv(path, sep=";")

# One row per unique combination of timestamp + soc_nom + is_rt should be one spectrum (28 freq points)
spectra = df.groupby(["timestamp_s", "soc_nom", "is_rt"]).size()
print("Number of distinct spectra in this file:", len(spectra))
print(spectra.describe())   # should cluster near 28 if grouping is right

# Does timestamp_s increase monotonically? (i.e. can it order check-ups in time)
print("\nUnique timestamps, in order:")
print(sorted(df["timestamp_s"].unique())[:10])

# Does cyc_charged change per check-up, or is it constant?
print("\ncyc_charged unique values:", df["cyc_charged"].unique())


Number of distinct spectra in this file: 291
count    291.0
mean      29.0
std        0.0
min       29.0
25%       29.0
50%       29.0
75%       29.0
max       29.0
dtype: float64

Unique timestamps, in order:
[np.float64(1665653927.0), np.float64(1665659081.0), np.float64(1665669260.0), np.float64(1665679641.0), np.float64(1665689716.0), np.float64(1665700156.0), np.float64(1665736717.0), np.float64(1665747274.0), np.float64(1665757709.0), np.float64(1665770490.0)]

cyc_charged unique values: [1 0]


In [2]:
import pandas as pd

# unique spectrum timestamps, sorted
timestamps = sorted(df["timestamp_s"].unique())

# convert to human-readable dates
dates = pd.to_datetime(timestamps, unit="s")
print("First 10 dates:")
print(dates[:10])

# gap between each spectrum and the next, in hours
gaps_hours = pd.Series(timestamps).diff() / 3600
print("\nGap between consecutive spectra (hours):")
print(gaps_hours.describe())

# how many gaps are "big" (more than 5 days = likely a new check-up)?
big_gaps = gaps_hours[gaps_hours > 24*5]
print("\nNumber of gaps > 5 days:", len(big_gaps))
print(big_gaps.head(10))

First 10 dates:
DatetimeIndex(['2022-10-13 09:38:47', '2022-10-13 11:04:41',
               '2022-10-13 13:54:20', '2022-10-13 16:47:21',
               '2022-10-13 19:35:16', '2022-10-13 22:29:16',
               '2022-10-14 08:38:37', '2022-10-14 11:34:34',
               '2022-10-14 14:28:29', '2022-10-14 18:01:30'],
              dtype='datetime64[s]', freq=None)

Gap between consecutive spectra (hours):
count    290.000000
mean      49.208415
std      148.121557
min        1.161667
25%        1.279167
50%        1.400417
75%        2.500833
max      982.888889
dtype: float64

Number of gaps > 5 days: 28
11     125.287222
21     471.025000
31     480.246667
46     460.407778
56     478.398333
66     481.690556
76     481.829444
86     483.258889
96     484.512500
106    480.661111
dtype: float64


In [6]:
def assign_cu_index(timestamps: pd.Series, gap_threshold_hours: float = 24.0) -> pd.Series:
    """
    Assigns a check-up (CU) index to each timestamp based on time gaps.

    timestamps: raw timestamp_s values (seconds since epoch), one per spectrum.
    gap_threshold_hours: any gap larger than this marks the start of a new CU.
    Returns: integer Series, same length/order as input, giving CU index (0,1,2,...).
    """
    ts = timestamps.reset_index(drop=True)
    order = ts.sort_values().index          # chronological order of the original positions
    sorted_ts = ts.loc[order]

    gap_seconds = sorted_ts.diff()
    is_new_cu = gap_seconds > (gap_threshold_hours * 3600)
    is_new_cu.iloc[0] = False               # first spectrum starts CU 0, not a "new" one

    cu_in_sorted_order = is_new_cu.cumsum()

    # map back onto the original (unsorted) order
    cu_index = pd.Series(index=order, data=cu_in_sorted_order.values).sort_index()
    return cu_index

In [4]:
cu_index = assign_cu_index(pd.Series(timestamps))
print(cu_index.value_counts().sort_index())

0     11
1     10
2     10
3     15
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11     5
12     5
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21    10
22    10
23    10
24    10
25    10
26    10
27    10
28    10
29     5
Name: count, dtype: int64


In [7]:
sorted_ts = pd.Series(timestamps).sort_values().reset_index(drop=True)
gap_hours = sorted_ts.diff() / 3600

# print index, gap-to-previous, and date, for anything near our suspicious CUs
inspect_df = pd.DataFrame({
    "timestamp": sorted_ts,
    "date": pd.to_datetime(sorted_ts, unit="s"),
    "gap_hours": gap_hours
})
print(inspect_df.iloc[20:45])   # around where CU 3 (15 spectra) would sit
print(inspect_df.iloc[125:150]) # around where CU 11/12 (5+5) would sit


       timestamp                date   gap_hours
20  1.666341e+09 2022-10-21 08:34:45    2.560000
21  1.668037e+09 2022-11-09 23:36:15  471.025000
22  1.668045e+09 2022-11-10 01:53:57    2.295000
23  1.668054e+09 2022-11-10 04:26:06    2.535833
24  1.668063e+09 2022-11-10 06:43:40    2.292778
25  1.668068e+09 2022-11-10 08:14:02    1.506111
26  1.668095e+09 2022-11-10 15:37:29    7.390833
27  1.668103e+09 2022-11-10 18:01:14    2.395833
28  1.668112e+09 2022-11-10 20:18:52    2.293889
29  1.668120e+09 2022-11-10 22:41:07    2.370833
30  1.668127e+09 2022-11-11 00:33:08    1.866944
31  1.669856e+09 2022-12-01 00:47:56  480.246667
32  1.669863e+09 2022-12-01 02:54:00    2.101111
33  1.669870e+09 2022-12-01 04:53:38    1.993889
34  1.669878e+09 2022-12-01 06:55:42    2.034444
35  1.669883e+09 2022-12-01 08:15:06    1.323333
36  1.669918e+09 2022-12-01 18:04:50    9.828889
37  1.669925e+09 2022-12-01 20:05:04    2.003889
38  1.669932e+09 2022-12-01 22:00:06    1.917222
39  1.669939e+09 202

In [8]:
# recompute in sorted (chronological) order for clarity
sorted_ts = pd.Series(timestamps).sort_values().reset_index(drop=True)
gap_hours = sorted_ts.diff() / 3600
is_new_cu = gap_hours > 24
is_new_cu.iloc[0] = False
cu_sorted = is_new_cu.cumsum()

inspect_df = pd.DataFrame({
    "timestamp": sorted_ts,
    "date": pd.to_datetime(sorted_ts, unit="s"),
    "gap_hours": gap_hours,
    "cu": cu_sorted
})

# find exact row range for each suspicious CU, with a little padding around it
for target_cu in [2, 3, 4, 10, 11, 12, 13, 28, 29]:
    idx = cu_sorted[cu_sorted == target_cu].index
    print(f"--- CU {target_cu}: rows {idx.min()}–{idx.max()}, count={len(idx)} ---")
    print(inspect_df.loc[max(idx.min()-1, 0) : idx.max()+1])
    print()


--- CU 2: rows 21–30, count=10 ---
       timestamp                date   gap_hours  cu
20  1.666341e+09 2022-10-21 08:34:45    2.560000   1
21  1.668037e+09 2022-11-09 23:36:15  471.025000   2
22  1.668045e+09 2022-11-10 01:53:57    2.295000   2
23  1.668054e+09 2022-11-10 04:26:06    2.535833   2
24  1.668063e+09 2022-11-10 06:43:40    2.292778   2
25  1.668068e+09 2022-11-10 08:14:02    1.506111   2
26  1.668095e+09 2022-11-10 15:37:29    7.390833   2
27  1.668103e+09 2022-11-10 18:01:14    2.395833   2
28  1.668112e+09 2022-11-10 20:18:52    2.293889   2
29  1.668120e+09 2022-11-10 22:41:07    2.370833   2
30  1.668127e+09 2022-11-11 00:33:08    1.866944   2
31  1.669856e+09 2022-12-01 00:47:56  480.246667   3

--- CU 3: rows 31–45, count=15 ---
       timestamp                date   gap_hours  cu
30  1.668127e+09 2022-11-11 00:33:08    1.866944   2
31  1.669856e+09 2022-12-01 00:47:56  480.246667   3
32  1.669863e+09 2022-12-01 02:54:00    2.101111   3
33  1.669870e+09 2022-12-01 

In [12]:
# build a lookup: timestamp -> CU index
ts_to_cu = pd.Series(cu_sorted.values, index=sorted_ts.values)

# map onto every row of the full raw dataframe (via timestamp_s)
df["cu"] = df["timestamp_s"].map(ts_to_cu)

# now this works — same length as df
cu3_rows = df[df["cu"] == 3]
print(cu3_rows[["soc_nom", "is_rt", "cyc_charged"]].drop_duplicates())

      soc_nom  is_rt  cyc_charged
899        10      1            1
928        30      1            1
957        50      1            1
986        70      1            1
1015       90      1            1
1044       90      0            0
1073       70      0            0
1102       50      0            0
1131       30      0            0
1160       10      0            0


In [2]:
import pandas as pd

path = "../data/raw/10.35097-1969/10.35097-1969/data/dataset/cell_eisv2/cell_eisv2_P001_1_S01_C10.csv"
df = pd.read_csv(path, sep=";")

timestamps = sorted(df["timestamp_s"].unique())

sorted_ts = pd.Series(timestamps).sort_values().reset_index(drop=True)
gap_hours = sorted_ts.diff() / 3600
is_new_cu = gap_hours > 24
is_new_cu.iloc[0] = False
cu_sorted = is_new_cu.cumsum()

# map CU index back onto every row of df
ts_to_cu = pd.Series(cu_sorted.values, index=sorted_ts.values)
df["cu"] = df["timestamp_s"].map(ts_to_cu)


In [3]:
cu3 = df[df["cu"] == 3]

combo_counts = cu3.groupby(["soc_nom", "is_rt", "cyc_charged"])["timestamp_s"].nunique()
print(combo_counts)

duplicated_combos = combo_counts[combo_counts > 1].index
for soc, rt, chg in duplicated_combos:
    subset = cu3[(cu3["soc_nom"] == soc) & (cu3["is_rt"] == rt) & (cu3["cyc_charged"] == chg)]
    print(f"\nsoc={soc}, is_rt={rt}, cyc_charged={chg}:")
    print(subset.groupby("timestamp_s")["valid"].agg(["min", "max", "count"]))


soc_nom  is_rt  cyc_charged
10       0      0              2
         1      1              1
30       0      0              2
         1      1              1
50       0      0              2
         1      1              1
70       0      0              2
         1      1              1
90       0      0              2
         1      1              1
Name: timestamp_s, dtype: int64

soc=10, is_rt=0, cyc_charged=0:
              min  max  count
timestamp_s                  
1.669947e+09    1    1     29
1.670006e+09    1    1     29

soc=30, is_rt=0, cyc_charged=0:
              min  max  count
timestamp_s                  
1.669939e+09    1    1     29
1.670002e+09    1    1     29

soc=50, is_rt=0, cyc_charged=0:
              min  max  count
timestamp_s                  
1.669932e+09    1    1     29
1.669994e+09    1    1     29

soc=70, is_rt=0, cyc_charged=0:
              min  max  count
timestamp_s                  
1.669925e+09    1    1     29
1.669987e+09    1    1     2

In [7]:
def assign_cu_index(timestamps: pd.Series, gap_threshold_hours: float = 24.0) -> pd.Series:
    """
    Assigns a check-up (CU) index to each timestamp based on time gaps.

    timestamps: raw timestamp_s values (seconds since epoch), one per spectrum.
    gap_threshold_hours: any gap larger than this marks the start of a new CU.
    Returns: integer Series, same length/order as input, giving CU index (0,1,2,...).
    """
    ts = timestamps.reset_index(drop=True)
    order = ts.sort_values().index
    sorted_ts = ts.loc[order]

    gap_seconds = sorted_ts.diff()
    is_new_cu = gap_seconds > (gap_threshold_hours * 3600)
    is_new_cu.iloc[0] = False

    cu_in_sorted_order = is_new_cu.cumsum()

    cu_index = pd.Series(index=order, data=cu_in_sorted_order.values).sort_index()
    return cu_index


cu_index_v2 = assign_cu_index(pd.Series(timestamps), gap_threshold_hours=48.0)
print(cu_index_v2.value_counts().sort_index())

0     11
1     10
2     10
3     15
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
12    10
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21    10
22    10
23    10
24    10
25    10
26    10
27    10
28     5
Name: count, dtype: int64


In [9]:
import pandas as pd
import glob, os

RAW_DIR = "../data/raw/10.35097-1969/10.35097-1969/data/dataset/cell_eisv2"

def assign_cu_index(timestamps: pd.Series, gap_threshold_hours: float = 48.0) -> pd.Series:
    ts = timestamps.reset_index(drop=True)
    if len(ts) == 0:
        return pd.Series([], dtype=int)
    order = ts.sort_values().index
    sorted_ts = ts.loc[order]
    # first element's diff is NaN -> fillna(False) makes it "not a new CU". Robust, no iloc.
    is_new = (sorted_ts.diff() > gap_threshold_hours * 3600).fillna(False)
    cu = pd.Series(index=order, data=is_new.cumsum().values).sort_index()
    return cu

all_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
sample = all_files[:1] + all_files[12:14] + all_files[40:42] + all_files[100:102] + all_files[-2:]

THRESH = 48.0
print(f"{'file':<40}{'rows':>7}{'#spec':>6}{'#CU':>5}{'odd':>5}{'max_within_h':>13}{'min_between_h':>14}")
print("-" * 90)
for f in sample:
    df = pd.read_csv(f, sep=";")
    ts = pd.Series(sorted(df["timestamp_s"].unique()))

    if len(ts) == 0:
        print(f"{os.path.basename(f):<40}{len(df):>7}{'0':>6}{'-':>5}{'-':>5}{'EMPTY / no timestamps':>27}")
        continue

    sorted_ts = ts.sort_values().reset_index(drop=True)
    gaps_h = sorted_ts.diff() / 3600
    cu_sorted = assign_cu_index(sorted_ts, THRESH).reset_index(drop=True)
    is_between = cu_sorted.diff() > 0
    within = gaps_h[~is_between].dropna()
    between = gaps_h[is_between].dropna()

    cu = assign_cu_index(ts, THRESH)
    odd = (cu.value_counts() != 10).sum()

    print(f"{os.path.basename(f):<40}{len(df):>7}{len(ts):>6}{cu.nunique():>5}{odd:>5}"
          f"{within.max():>13.1f}{(between.min() if len(between) else float('nan')):>14.1f}")

file                                       rows #spec  #CU  odd max_within_h min_between_h
------------------------------------------------------------------------------------------
cell_eisv2_P000_0_S20_C00.csv                 0     0    -    -      EMPTY / no timestamps
cell_eisv2_P001_1_S01_C10.csv              8439   291   29    3         31.7         125.3
cell_eisv2_P001_2_S04_C09.csv              8410   290   29    2         31.4         128.4
cell_eisv2_P010_2_S12_C10.csv              8410   290   29    2         32.5         127.7
cell_eisv2_P010_3_S13_C10.csv              8410   290   29    2         32.4         128.1
cell_eisv2_P030_2_S08_C05.csv              2784    96    9    2         14.6         127.0
cell_eisv2_P030_3_S09_C05.csv              2784    96    9    2         12.9         127.9
cell_eisv2_P076_2_S16_C02.csv              8468   292   29    3         29.5         127.5
cell_eisv2_P076_3_S17_C02.csv              8439   291   29    3         29.3         128.9

In [10]:
import glob, os
import pandas as pd

all_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Total files: {len(all_files)}")

empty, nonempty = [], []
for f in all_files:
    try:
        n = len(pd.read_csv(f, sep=";", usecols=["timestamp_s"]))
    except Exception:
        n = 0
    (empty if n == 0 else nonempty).append(os.path.basename(f))

print(f"Non-empty (real cells): {len(nonempty)}")
print(f"Empty files: {len(empty)}")
print("\nEmpty file names:")
for name in empty:
    print("  ", name)


Total files: 240
Non-empty (real cells): 228
Empty files: 12

Empty file names:
   cell_eisv2_P000_0_S20_C00.csv
   cell_eisv2_P000_0_S20_C01.csv
   cell_eisv2_P000_0_S20_C02.csv
   cell_eisv2_P000_0_S20_C03.csv
   cell_eisv2_P000_0_S20_C04.csv
   cell_eisv2_P000_0_S20_C05.csv
   cell_eisv2_P000_0_S20_C06.csv
   cell_eisv2_P000_0_S20_C07.csv
   cell_eisv2_P000_0_S20_C08.csv
   cell_eisv2_P000_0_S20_C09.csv
   cell_eisv2_P000_0_S20_C10.csv
   cell_eisv2_P000_0_S20_C11.csv


In [1]:
import sys
sys.path.insert(0, "..")
from src.prep.load_eis import process_cell_file

path = "../data/raw/10.35097-1969/10.35097-1969/data/dataset/cell_eisv2/cell_eisv2_P001_1_S01_C10.csv"
out = process_cell_file(path)

print("shape:", out.shape)
print("check-ups:", out["cu_index"].nunique())
spectra = out.groupby(["cu_index", "soc_nom", "is_rt"])
print("number of spectra:", spectra.ngroups)
print("points per spectrum (should all be 28):")
print(spectra.size().value_counts())
out.head()


shape: (7980, 14)
check-ups: 29
number of spectra: 285
points per spectrum (should all be 28):
28    285
Name: count, dtype: int64


,cell_id,pack,replicate,board,channel,cu_index,soc_nom,is_rt,freq_Hz,Z_real_mOhm,Z_imag_mOhm,temp_degC,soh_imp,valid
0,cell_eisv2_P001_1_S01_C10,P001,1,S01,C10,0,10,1,10000.0000,13.758,-11.192,24.95,100.0,1
1,cell_eisv2_P001_1_S01_C10,P001,1,S01,C10,0,10,1,6756.7568,13.839,-7.921,24.95,100.0,1
2,cell_eisv2_P001_1_S01_C10,P001,1,S01,C10,0,10,1,5000.0000,13.997,-5.765,24.95,100.0,1
3,cell_eisv2_P001_1_S01_C10,P001,1,S01,C10,0,10,1,3125.0000,14.177,-3.342,24.95,100.0,1
4,cell_eisv2_P001_1_S01_C10,P001,1,S01,C10,0,10,1,2083.3333,14.380,-1.773,24.95,100.0,1


In [1]:
import sys
sys.path.insert(0, "..")
from src.prep.load_eis import load_all_cells

RAW = "../data/raw/10.35097-1969/10.35097-1969/data/dataset/cell_eisv2"
combined, skipped = load_all_cells(RAW, save_path="../data/interim/eis_spectra.parquet")

print("cells:", combined["cell_id"].nunique())
print("skipped (empty files):", len(skipped))
print("total rows:", len(combined))

spectra = combined.groupby(["cell_id", "cu_index", "soc_nom", "is_rt"])
print("total spectra:", spectra.ngroups)
print("points per spectrum (should be all 28):")
print(spectra.size().value_counts())

cells: 228
skipped (empty files): 12
total rows: 1102304
total spectra: 39368
points per spectrum (should be all 28):
28    39368
Name: count, dtype: int64


In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)   # show all 29 columns
pd.set_option("display.width", 250)

labeled = pd.read_parquet("../data/interim/eis_labeled.parquet")
labeled.sample(20)        # 20 random rows as a table

,cell_id,pack,replicate,board,channel,cu_index,timestamp_s,soc_nom,is_rt,freq_Hz,Z_real_mOhm,Z_imag_mOhm,temp_degC,soh_imp,z_ref_init_mOhm,z_ref_now_mOhm,valid,param_id,aging_type,temp_cat,soc_idle_cat,soc_limits_cat,c_rate_cat,profile,temp_setpoint_degC,cell_key,cap_timestamp_s,soh_cap,cap_aged_est_Ah
943669,cell_eisv2_P075_2_S18_C02,P075,2,S18,C02,22,1.704350e+09,50,1,675.6757,16.159,0.393,24.85,87.115401,19.129,24.059,1,75,Profile,D,-,C,A,A,40,P075_2_S18_C02,1.704340e+09,57.600754,2.364011
910081,cell_eisv2_P038_2_S07_C07,P038,2,S07,C07,20,1.700775e+09,10,0,312.5000,29.286,4.342,10.16,70.544875,35.423,56.291,1,38,Cyclic,B,-,C,B,-,10,P038_2_S07_C07,1.700680e+09,29.250893,1.938763
469471,cell_eisv2_P001_1_S01_C10,P001,1,S01,C10,8,1.678969e+09,90,0,675.6757,17.245,1.620,1.46,100.989878,41.067,40.254,1,1,Calendar,A,A,-,-,-,0,P001_1_S01_C10,1.678915e+09,91.844511,2.877668
179222,cell_eisv2_P075_3_S19_C02,P075,3,S19,C02,2,1.668120e+09,30,0,14.7059,15.758,0.252,40.13,99.752402,16.511,16.593,1,75,Profile,D,-,C,A,A,40,P075_3_S19_C02,1.668038e+09,84.648132,2.769722
660905,cell_eisv2_P068_1_S06_C03,P068,1,S06,C03,13,1.687996e+09,50,1,0.5000,27.030,1.759,25.13,81.516171,19.325,26.469,1,68,Profile,B,-,B,A,A,10,P068_1_S06_C03,1.687986e+09,51.284772,2.269272
127577,cell_eisv2_P044_3_S13_C00,P044,3,S13,C00,1,1.666346e+09,10,0,2083.3333,14.032,-1.968,25.63,102.429473,22.766,21.660,1,44,Cyclic,C,-,A,D,-,25,P044_3_S13_C00,1.666230e+09,77.983779,2.669757
791866,cell_eisv2_P050_1_S10_C09,P050,1,S10,C09,17,1.695243e+09,10,1,3.1250,22.418,2.268,24.70,102.018249,24.261,23.281,1,50,Cyclic,C,-,C,B,-,25,P050_1_S10_C09,1.695240e+09,66.483355,2.497250
618506,cell_eisv2_P013_2_S16_C10,P013,2,S16,C10,12,1.686189e+09,10,1,6756.7568,13.866,-8.552,25.05,100.199127,23.399,23.551,1,13,Calendar,D,A,-,-,-,40,P013_2_S16_C10,1.686185e+09,85.683602,2.785254
22150,cell_eisv2_P070_1_S06_C02,P070,1,S06,C02,0,1.665691e+09,70,1,1470.5883,14.602,-1.072,24.91,100.000000,19.496,19.496,1,70,Profile,B,-,C,D,B,10,P070_1_S06_C02,1.665624e+09,96.362847,2.945443
41974,cell_eisv2_P073_1_S11_C02,P073,1,S11,C02,0,1.665743e+09,70,0,3.1250,18.480,0.356,25.46,100.000000,19.223,19.223,1,73,Profile,C,-,C,D,B,25,P073_1_S11_C02,1.665625e+09,97.239971,2.958600
